# Proof of concept

This notebook is reponsible that a model can learn from this specific feature vector to drive in a track.

In [1]:
from deepracer_genesis.randomization.catalog import CATALOG, BY_NAME, by_layer
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
from deepracer_genesis.experiment import (
    AsymmetricCameraPolicy,
    CameraEnvironment,
    FeatureEnvironment,
    Evaluation,
    PPO
)

from deepracer_genesis.randomization.spaces import FloatRange, IntRange, Choice
from deepracer_genesis.tools.zoo import view_zoo

import optuna

## Proof of Camera Training

In this section we train a simple model to learn on camera mode just to prove this system can do the same as the old deepracer system.

We first define the HPO search then a simple camera enviroment

In [4]:
SEED = 0
ROOT = "runs/best_camera"
WALL_BUDGET_H = 10.0
REUSE = False                      # return recorded stage results from cache

# renderer bench
BENCH_STEPS = 150

# HPO (no-DR, single track: measures pure learning ability)
HPO_TRIALS = 16
HPO_DEADLINE_H = 3.4
TRIAL_STEPS = 1_200_000
TRIAL_EVAL_EVERY = 600_000
TRIAL_TRACK = "reinvent_base"

# final training (winner config + zoo + full DR)
FINAL_STEPS_MAX = 30_000_000
FINAL_STEPS_MIN = 2_000_000
FINAL_RESERVE_H = 1.0
FINAL_EVALS = 6

RESOLUTION = np.array((160, 120))            # physical-camera parity

NUM_ENVS = 1024

In [1]:
SEARCH_SPACE = {
    "lr": FloatRange(1e-4, 1e-2, log=True),
    "entropy_coef": FloatRange(1e-3, 3e-2, log=True),
    "epochs": IntRange(3, 8),
    "clip": FloatRange(0.1, 0.3)
}


def suggest_architecture(trial: optuna.Trial) -> dict:
    """
    Search space for the architecture

    trial: Optuna Trial.
    """
    depth = trial.suggest_int("depth",2,6)
    width = trial.suggest_categorical("width", [128,256,521])
    return tuple(max(width // 2**i, 32) for i in range(depth))

NameError: name 'FloatRange' is not defined

In [ ]:
def suggest_architecture(trial: optuna.Trial) -> dict:
    resolutions = [
        tuple(RESOLUTION * i)
        for i in range(1, 4)
    ]
    suggestions = {
        "name": space.suggest(trial,name)
        for name, space in SEARCH_SPACE.items()
    }
    
    return (
        CameraEnvironment(
            num_envs=NUM_ENVS,
            tracks=[TRIAL_TRACK],
            resolution=trial.suggest_categorical("resolution", resolutions)
        ) >> AsymmetricCameraPolicy(
            actor_keys=("camera",),
            critic_keys=("camera"),
            cnn = {
                "cha"
            }
        ) >> PPO(
            lr = suggestions["lr"],
            entropy_coef= suggestions["entropy_coef"],
            epochs=suggestions["epochs"],
            clip=suggestions["clip"]
        )
    )